# [OR-2 Term Project] 미디어관 엘리베이터 대기 시스템 분석
## M/G/s/K Batch Queueing Model & Decision Tree 경제성 분석
**팀원 C (종원)** — Gurobi 큐잉 모델링 수식 풀이

### 분석 목적
팀원 A(민섭), B(채윤)가 도출한 파라미터를 토대로:
1. M/G/s/K in batch units 균형방정식 풀이 → 정상상태 확률 p_n 도출
2. 평균 대기시간 W_q 계산
3. 현행(s=3) vs 개선안(s=4) 성능 비교
4. Decision Tree & 민감도 분석 → 최적 운영 대수 제언

### 모델 파라미터 출처
| 파라미터 | 출처 | 값 |
|---|---|---|
| λ, μ, K, B | 팀원 A (민섭) `service_time_params.json` | 4.3556, 0.239, 14, 17 |
| bk, T, X | 팀원 B (채윤) `survey_params.json` | 0.9807, 5.49, 4.06 |

### 분석 흐름
```
[파라미터 로드] → [균형방정식 Gurobi 풀이] → [W_q 계산]
→ [s=3 vs s=4 비교] → [Decision Tree] → [결과 저장]
```

In [54]:
import json
import numpy as np
import os

# ---- 경로 설정 ----
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
REPO_ROOT = os.path.dirname(NOTEBOOK_DIR)
OUT_DIR = os.path.join(REPO_ROOT, "outputs")

# ---- 파라미터 로드 ----
with open(os.path.join(OUT_DIR, "service_time_params.json"), encoding="utf-8") as f:
    svc = json.load(f)

with open(os.path.join(OUT_DIR, "survey_params.json"), encoding="utf-8") as f:
    srv = json.load(f)

# ---- 모델 입력값 세팅 ----
lam = svc["lambda_per_min"]       # 도착률 λ = 4.3556 명/분
mu  = svc["mu_per_min_per_car"]   # 서비스율 μ = 0.239 cycles/분 (호기 1대)
K   = svc["K_capacity"]           # 대기 상한 K = 14 (실측 세션별 최대 대기 평균)
B   = svc["batch_size_B"]         # 배치 크기 B = 17 (정원 1150kg, Full-batch 가정)
bk  = srv["bk"]                   # 이탈 감쇠 상수 bk = 0.9807
T   = srv["T_avg_min"]            # 마지노선 대기시간 T = 5.49분
X   = srv["X_avg_score"]          # 불만족도 X = 4.06점

print("=" * 45)
print("  모델 입력 파라미터 확인")
print("=" * 45)
print(f"  λ (도착률)          = {lam} 명/분")
print(f"  μ (서비스율, 1대)   = {mu} cycles/분")
print(f"  K (대기 상한)       = {K} 명")
print(f"  B (배치 크기)       = {B} 명")
print(f"  bk (이탈 감쇠)      = {bk}")
print(f"  T (마지노선)        = {T} 분")
print(f"  X (불만족도)        = {X} 점")
print("=" * 45)

  모델 입력 파라미터 확인
  λ (도착률)          = 4.3556 명/분
  μ (서비스율, 1대)   = 0.239 cycles/분
  K (대기 상한)       = 14 명
  B (배치 크기)       = 17 명
  bk (이탈 감쇠)      = 0.9807
  T (마지노선)        = 5.49 분
  X (불만족도)        = 4.06 점


## 2단계 — 균형방정식 풀이 (Gurobi)

### 모델 구조: M/G/s/K in Batch Units

**상태 정의**
- 상태 n = 현재 시스템 내 총 인원 (대기 + 서비스 중)
- n = 0 ~ K+c (c = s×B, 서비스 중 최대 인원)

**상태 의존적 도착률 λ_n**
- n < c (서비스 중 자리 여유 있음) → λ_n = λ
- n ≥ c (대기열 발생) → λ_n = λ · bk^(n-c)
- n = N-1 (시스템 꽉 참) → λ_n = 0

**상태 의존적 서비스율 μ_n**
- n = 0 → μ_n = 0 (아무도 없음)
- n > 0 → μ_n = min(s, ceil(n/B)) × μ (운행 중인 대수 × 1대 서비스율)

**Little's Law**
- L = Σ n·p_n
- Lq = Σ max(n-c, 0)·p_n  
- Wq = Lq / λ_eff

In [55]:
import math
import numpy as np
import gurobipy as gp
from gurobipy import GRB

def solve_queueing(lam, mu, K, B, s, bk):
    """
    M/G/s/K in Batch Units 균형방정식 풀이
    Gurobi 잔차 최소화로 정상상태 확률 도출
    
    상태 n = 시스템 내 총 인원 (0 ~ K+c)
    도착: 1명씩 (n → n+1)
    서비스: B명씩 한꺼번에 (n → n-B)
    
    Parameters:
        lam : 도착률 λ (명/분)
        mu  : 서비스율 μ (cycles/분, 호기 1대)
        K   : 대기 상한 (명)
        B   : 배치 크기 (정원, 1대 기준)
        s   : 엘리베이터 대수
        bk  : 이탈 감쇠 상수
    """
    c = s * B
    N = K + c + 1

    def lam_n(n):
        if n >= N - 1:
            return 0.0
        if n < c:
            return lam
        return lam * (bk ** (n - c))

    def mu_n(n):
        if n <= 0:
            return 0.0
        return min(s, math.ceil(n / B)) * mu

    # ---- 전이율 행렬 Q 구성 ----
    Q = np.zeros((N, N))
    for n in range(N):
        if n < N - 1:
            Q[n][n+1] = lam_n(n)
        if n >= B:
            Q[n][n-B] = mu_n(n)
        Q[n][n] = -(lam_n(n) + mu_n(n))

    # A = Q^T, 마지막 행 정규화로 교체
    A = Q.T.copy()
    A[-1, :] = 1.0
    b_vec = np.zeros(N)
    b_vec[-1] = 1.0

    # ---- Gurobi 잔차 최소화 ----
    model = gp.Model("MG_sK_batch")
    model.setParam("OutputFlag", 0)

    p = model.addVars(N, lb=0.0, ub=1.0, name="p")
    r = model.addVars(N, lb=-GRB.INFINITY, name="r")

    # 균형방정식: A*p = b (잔차 r로 표현)
    for i in range(N):
        lhs = gp.quicksum(A[i][j] * p[j] for j in range(N))
        model.addConstr(lhs - r[i] == b_vec[i])

    # 정규화
    model.addConstr(gp.quicksum(p[n] for n in range(N)) == 1.0)

    # 잔차 최소화 (균형방정식 오차 최소화)
    model.setObjective(
        gp.quicksum(r[i] * r[i] for i in range(N)),
        GRB.MINIMIZE
    )
    model.optimize()

    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Gurobi 최적해를 찾지 못했습니다. 상태: {model.Status}")

    prob = np.array([p[n].X for n in range(N)])
    prob = np.clip(prob, 0, None)
    prob = prob / prob.sum()

    # ---- 성능 지표 ----
    lam_eff = sum(lam_n(n) * prob[n] for n in range(N))
    L  = sum(n * prob[n] for n in range(N))
    Lq = sum(max(n - c, 0) * prob[n] for n in range(N))
    W  = L  / lam_eff if lam_eff > 0 else 0
    Wq = Lq / lam_eff if lam_eff > 0 else 0
    P_loss = prob[N-1]

    return {
        "s": s, "c": c, "N_states": N,
        "prob": prob, "p0": prob[0], "pK": prob[N-1],
        "L": L, "Lq": Lq, "W": W, "Wq": Wq,
        "lam_eff": lam_eff, "P_loss": P_loss,
    }

print("✓ M/G/s/K in Batch Units 균형방정식 함수 정의 완료 (Gurobi)")
print(f"  s=3: c={3*B}명, N={K+3*B+1}개 상태")
print(f"  s=4: c={4*B}명, N={K+4*B+1}개 상태")

✓ M/G/s/K in Batch Units 균형방정식 함수 정의 완료 (Gurobi)
  s=3: c=51명, N=66개 상태
  s=4: c=68명, N=83개 상태


## 3단계 — s=3 vs s=4 비교
현행(엘리베이터 3대) vs 개선안(직원용 추가 개방 4대) 성능 비교

In [56]:
# ---- s=3 (현행) vs s=4 (개선안) 계산 ----
result_s3 = solve_queueing(lam, mu, K, B, s=3, bk=bk)
result_s4 = solve_queueing(lam, mu, K, B, s=4, bk=bk)

print("=" * 55)
print("  현행(s=3) vs 개선안(s=4) 비교")
print("=" * 55)
print(f"{'항목':<20} {'s=3 (현행)':>15} {'s=4 (개선안)':>15}")
print("-" * 55)
print(f"{'평균 대기시간 Wq':<20} {result_s3['Wq']:>13.4f}분 {result_s4['Wq']:>13.4f}분")
print(f"{'시스템 내 평균인원 L':<20} {result_s3['L']:>13.4f}명 {result_s4['L']:>13.4f}명")
print(f"{'대기 평균인원 Lq':<20} {result_s3['Lq']:>13.4f}명 {result_s4['Lq']:>13.4f}명")
print(f"{'실질 도착률 λ_eff':<20} {result_s3['lam_eff']:>12.4f}/분 {result_s4['lam_eff']:>12.4f}/분")
print(f"{'손실 확률 P_loss':<20} {result_s3['P_loss']:>15.4f} {result_s4['P_loss']:>15.4f}")
print(f"{'p_0 (빈 확률)':<20} {result_s3['p0']:>15.4f} {result_s4['p0']:>15.4f}")
print(f"{'총 상태 수 N':<20} {result_s3['N_states']:>15} {result_s4['N_states']:>15}")
print("-" * 55)
print(f"\n  Wq 개선율    : {(1 - result_s4['Wq']/result_s3['Wq'])*100:.1f}% 감소")
print(f"  P_loss 개선율: {(1 - result_s4['P_loss']/result_s3['P_loss'])*100:.1f}% 감소")
print(f"  λ_eff 개선율 : {(result_s4['lam_eff']/result_s3['lam_eff'] - 1)*100:.1f}% 증가")

  현행(s=3) vs 개선안(s=4) 비교
항목                          s=3 (현행)       s=4 (개선안)
-------------------------------------------------------
평균 대기시간 Wq                  0.4164분        0.2567분
시스템 내 평균인원 L               30.2501명       33.1868명
대기 평균인원 Lq                  1.6323명        1.0430명
실질 도착률 λ_eff               3.9200/분       4.0636/분
손실 확률 P_loss                  0.0944          0.0645
p_0 (빈 확률)                    0.0014          0.0013
총 상태 수 N                          66              83
-------------------------------------------------------

  Wq 개선율    : 38.4% 감소
  P_loss 개선율: 31.7% 감소
  λ_eff 개선율 : 3.7% 증가


## 4단계 — Decision Tree 경제성 분석

### 분석 목적
직원용 엘리베이터 추가 개방(s=3 → s=4) 의사결정의 경제적 타당성 분석

### 구조
- **선택 A**: s=3 유지 (가동 비용 없음, 학생 불만족 비용 발생)
- **선택 B**: s=4 개방 (가동 비용 C_op 발생, 학생 불만족 비용 감소)

### 학생 불만족 비용 (Loss of Goodwill)
- 불만족도 X = 4.06점 (5점 만점, 채윤 설문)
- 마지노선 T = 5.49분 (학생 평균 인내 대기시간)
- P(Wq > T): Wq가 마지노선을 초과할 확률

### Break-even 분석
C_op* = s=3과 s=4의 불만족 비용 차이
→ C_op < C_op* 이면 s=4 운영이 경제적으로 유리

In [59]:
# ======================================================
# 4단계 — Decision Tree 경제성 분석
# ======================================================

# ---- 기본 설정 ----
peak_duration = 15  # 피크타임 지속시간 (분)
n_students = lam * peak_duration  # 피크타임 총 도착 학생 수

print("=" * 55)
print("  기본 설정")
print("=" * 55)
print(f"  피크타임 지속시간    = {peak_duration}분")
print(f"  총 도착 학생 수      = {n_students:.1f}명")
print(f"  마지노선 T           = {T}분")
print(f"  불만족도 X           = {X}점 (5점 만점)")

# ---- 불만족 비용 계산 ----
# 불만족 비용 = X × (Wq/T) × n_students
# Wq/T: 마지노선 대비 실제 대기시간 비율
# X: 불만족도 가중치
# n_students: 피크타임 영향받는 학생 수

Wq_s3 = result_s3["Wq"]
Wq_s4 = result_s4["Wq"]

cost_s3 = X * (Wq_s3 / T) * n_students
cost_s4 = X * (Wq_s4 / T) * n_students
cost_diff = cost_s3 - cost_s4  # s=4 개방 시 불만족 감소분 = Break-even C_op*

print(f"\n  Wq/T 비율 (s=3)      = {Wq_s3/T:.4f} ({Wq_s3/T*100:.1f}%)")
print(f"  Wq/T 비율 (s=4)      = {Wq_s4/T:.4f} ({Wq_s4/T*100:.1f}%)")
print(f"\n  불만족 비용 (s=3)    = {X} × {Wq_s3/T:.4f} × {n_students:.1f} = {cost_s3:.2f} 단위")
print(f"  불만족 비용 (s=4)    = {X} × {Wq_s4/T:.4f} × {n_students:.1f} = {cost_s4:.2f} 단위")
print(f"  불만족 감소분        = {cost_diff:.2f} 단위")

# ---- Break-even 분석 ----
print("\n" + "=" * 55)
print("  Break-even 분석")
print("=" * 55)
print(f"""
  [s=3 유지] 총 비용 = 0 (가동비) + {cost_s3:.2f} (불만족) = {cost_s3:.2f}
  [s=4 개방] 총 비용 = C_op     + {cost_s4:.2f} (불만족)

  s=4가 유리한 조건:
  C_op + {cost_s4:.2f} < {cost_s3:.2f}
  C_op < {cost_diff:.2f} 단위

  → 가동 비용이 {cost_diff:.2f} 단위 미만이면 s=4 개방이 경제적으로 타당
""")

# ---- Decision Tree 요약 ----
print("=" * 55)
print("  Decision Tree 결과 요약")
print("=" * 55)
print(f"  {'항목':<25} {'s=3 유지':>12} {'s=4 개방':>12}")
print(f"  {'-'*50}")
print(f"  {'Wq (분)':<25} {Wq_s3:>12.4f} {Wq_s4:>12.4f}")
print(f"  {'Wq/T 비율':<25} {Wq_s3/T:>12.4f} {Wq_s4/T:>12.4f}")
print(f"  {'불만족 비용':<25} {cost_s3:>12.2f} {cost_s4:>12.2f}")
print(f"  {'가동 비용':<25} {'0':>12} {'C_op':>12}")
print(f"  {'총 기대비용':<25} {cost_s3:>12.2f} {'C_op+'+str(round(cost_s4,2)):>12}")
print(f"  {'-'*50}")
print(f"\n  ✓ Break-even C_op* = {cost_diff:.2f} 단위")
print(f"  ✓ C_op < {cost_diff:.2f} 이면 → s=4 개방 권장")
print(f"  ✓ C_op ≥ {cost_diff:.2f} 이면 → s=3 유지 권장")

# ---- 민감도 분석 ----
print("\n" + "=" * 55)
print("  민감도 분석 (λ 변화에 따른 Break-even)")
print("=" * 55)
print(f"  {'λ (명/분)':<15} {'Wq s=3':>10} {'Wq s=4':>10} {'C_op*':>10}")
print(f"  {'-'*48}")

for lam_test in [3.4, 4.36, 4.67, 5.0]:
    r3 = solve_queueing(lam_test, mu, K, B, s=3, bk=bk)
    r4 = solve_queueing(lam_test, mu, K, B, s=4, bk=bk)
    n_stu = lam_test * peak_duration
    c3 = X * (r3["Wq"] / T) * n_stu
    c4 = X * (r4["Wq"] / T) * n_stu
    print(f"  {lam_test:<15} {r3['Wq']:>10.4f} {r4['Wq']:>10.4f} {c3-c4:>10.2f}")

  기본 설정
  피크타임 지속시간    = 15분
  총 도착 학생 수      = 65.3명
  마지노선 T           = 5.49분
  불만족도 X           = 4.06점 (5점 만점)

  Wq/T 비율 (s=3)      = 0.0758 (7.6%)
  Wq/T 비율 (s=4)      = 0.0468 (4.7%)

  불만족 비용 (s=3)    = 4.06 × 0.0758 × 65.3 = 20.12 단위
  불만족 비용 (s=4)    = 4.06 × 0.0468 × 65.3 = 12.40 단위
  불만족 감소분        = 7.72 단위

  Break-even 분석

  [s=3 유지] 총 비용 = 0 (가동비) + 20.12 (불만족) = 20.12
  [s=4 개방] 총 비용 = C_op     + 12.40 (불만족)

  s=4가 유리한 조건:
  C_op + 12.40 < 20.12
  C_op < 7.72 단위

  → 가동 비용이 7.72 단위 미만이면 s=4 개방이 경제적으로 타당

  Decision Tree 결과 요약
  항목                              s=3 유지       s=4 개방
  --------------------------------------------------
  Wq (분)                          0.4164       0.2567
  Wq/T 비율                         0.0758       0.0468
  불만족 비용                           20.12        12.40
  가동 비용                                0         C_op
  총 기대비용                           20.12    C_op+12.4
  --------------------------------------------------

  ✓ Break-even C_o